In [ ]:
import pandas as pd
import vivarium_inputs
import gbd_mapping
import xlsxwriter
from xlsxwriter.utility import xl_rowcol_to_cell

In [ ]:
from lsff_utils import config_utils

In [ ]:
workbook = xlsxwriter.Workbook("results_spreadsheet.xlsx")
sheet = workbook.add_worksheet("Model Results")

In [ ]:
location = "india"
vehicle = "rice"
intervention_scenario = "intervention"

In [ ]:
daly_categories = {
    "ntd": "NTD DALYs Averted by Folate Fortification",
    "anemia": "Anemia DALYs Averted by Iron Fortification",
    "lbwsg": "Neonatal DALYs Attributable to Low Birth Weight/Short Gestation Averted by Iron Fortification",
    "maternal_disorders": "Maternal Disorders DALYs Averted by Iron Fortification",
}

In [ ]:
header = []

for category in daly_categories.values():
    header += [f"{category} (1000s)", f"{category} (%)"]

header

In [ ]:
subtotals = {
    "DALYs Averted by Iron Fortification (1000s)": ["Anemia DALYs Averted by Iron Fortification (1000s)", "Neonatal DALYs Attributable to Low Birth Weight/Short Gestation Averted by Iron Fortification (1000s)"],
    "DALYs Averted by Fortification (1000s)": ["DALYs Averted by Iron Fortification (1000s)", "NTD DALYs Averted by Folate Fortification (1000s)"]
}

In [ ]:
header.insert(0, "DALYs Averted by Fortification (1000s)")
header.insert(3, "DALYs Averted by Iron Fortification (1000s)")
header

In [ ]:
num_sidebar_cols = 2
sheet.set_column(0, 0, 1)
sheet.set_column(1, 1, 10)

In [ ]:
header_format = workbook.add_format({"bold": True, "text_wrap": True})

for idx, header_value in enumerate(header):
    sheet.write(0, idx + num_sidebar_cols, header_value, header_format)
    sheet.set_column(idx + num_sidebar_cols, idx + num_sidebar_cols, 15)

In [ ]:
display_quintiles = {
    "lowest": "Poorest",
    "second": "Second",
    "middle": "Third",
    "fourth": "Fourth",
    "highest": "Wealthiest",
}

In [ ]:
next_row = 1

In [ ]:
comma_format = workbook.add_format({"num_format": '_(* #,##0_);_(* (#,##0);_(* "-"??_);_(@_)'})
percent_format = workbook.add_format({"num_format": '0.0%;-0.0%;"-"'})
summary_format = workbook.add_format({"bold": True, "italic": True})
bold_format = workbook.add_format({"bold": True})

In [ ]:
intervention_scenario_names = {
    "intervention_25_nrv": "25% NRV",
    "intervention_100_nrv": "100% NRV",
}

In [ ]:
for location, vehicle, intervention_scenario in config_utils.get_configured_combos(["location", "vehicle", "scenario"]):
    summary = f"{location.title()} -- {vehicle.title()}"
    if intervention_scenario != "intervention":
        summary += f" -- {intervention_scenario_names[intervention_scenario]} scenario"

    sheet.write(next_row, 0, summary, summary_format)
    next_row += 1

    sheet.write(next_row, 1, "National")
    national_row = next_row
    next_row += 1

    dalys_by_scenario = pd.read_csv(f'./results/{location}/{vehicle}/{intervention_scenario}/dalys_by_scenario.csv')
    dalys_by_scenario = dalys_by_scenario.set_index([c for c in dalys_by_scenario.columns if c != "value"]).value

    quintile_rows = []

    sheet.write(next_row, 0, "Wealth quintile", bold_format)
    next_row += 1

    for quintile, display_quintile in display_quintiles.items():
        sheet.write(next_row, 1, display_quintile)

        for entity, header_category in daly_categories.items():
            baseline_dalys = dalys_by_scenario.loc[("baseline", entity, quintile)]
            dalys_averted = baseline_dalys - dalys_by_scenario.loc[(intervention_scenario, entity, quintile)]

            sheet.write_number(next_row, header.index(f"{header_category} (1000s)") + num_sidebar_cols, dalys_averted.sum() / 1_000, comma_format)
            if baseline_dalys.sum() > 0:
                percent = dalys_averted.sum() / baseline_dalys.sum()
            else:
                percent = 0
            sheet.write_number(next_row, header.index(f"{header_category} (%)") + num_sidebar_cols, percent, percent_format)
        
        for subtotal_header, sum_headers in subtotals.items():
            formula = '+'.join([xl_rowcol_to_cell(next_row, header.index(h) + num_sidebar_cols) for h in sum_headers])
            sheet.write_formula(next_row, header.index(subtotal_header) + num_sidebar_cols, formula, comma_format)

        quintile_rows.append(next_row)
        next_row += 1

    for entity, header_category in daly_categories.items():
        baseline_dalys = dalys_by_scenario.loc[("baseline", entity)]
        dalys_averted = baseline_dalys - dalys_by_scenario.loc[(intervention_scenario, entity)]

        if baseline_dalys.sum() > 0:
            percent = dalys_averted.sum() / baseline_dalys.sum()
        else:
            percent = 0
        sheet.write_number(national_row, header.index(f"{header_category} (%)") + num_sidebar_cols, percent, percent_format)
    
    for header_value in header:
        if "1000s" in header_value:
            header_index = header.index(header_value) + num_sidebar_cols
            formula = '+'.join([xl_rowcol_to_cell(row, header_index) for row in quintile_rows])

            sheet.write_formula(national_row, header_index, formula, comma_format)
    
    next_row += 1

In [ ]:
dalys_by_scenario

In [ ]:
# sheet.autofit()

In [ ]:
workbook.close()